# 让数据驱动界面

> React 最大的变化，就是把“操作页面”变成了“改变数据/值”。页面根据数据自动长成应该有的样子，这就是 React 的另一个核心思想：数据驱动界面。

## 前言：回顾上一节的两个组件

PageHeading 组件把标题 title 和副标题 subtitle 留成空位，由调用它的页面传入内容，因此同一个组件可以在个人主页和文字实验室显示不同文案。

App 组件有两个职责：

- 收纳现有的两个页面组件：HomePage 和 TextLabPage。
- 根据当前页面的值，决定显示哪个页面。

这两个组件已经可以工作，但它们暴露出一个更重要的问题：界面不应该把所有内容和判断都直接写死在标签中。

## 数据与界面分离

首页的“关于我”直接写在 HomePage 的 JSX 中，修改文案就必须在一堆标签里寻找文字，这就是数据和界面没有分离。更好的做法是把网站要显示的内容集中在一个数据文件中。

### src/data/site.js 用于存放数据

js 是 JavaScript 模块文件它和 JSON 很相似。不同点有：最后一项可以保留尾逗号，可以使用 export。

组件只负责排版、样式和结构，`site.js` 只负责内容。

## 状态驱动 UI

上面的内容是从外部传进组件的固定数据。另一类数据会在用户操作时自己变化：它有默认值，组件自己保存和管理，值一变，界面自动跟着变。在 React 中，这种值叫组件的状态（state）。

状态有四个特点：

- 它是组件自己的一部分，不是父组件临时喂进来的。
- 它有一个默认值。
- 组件显示什么，取决于这个值。
- 值允许被更新，更新后界面自动重新渲染。

### 文字实验室中的例子

在文字实验室输入框下方有一行“已输入 N 字”。输入、删除文字时，数字会立即变化：输入框当前文字是 state，字数是从 state 计算出来的 UI。

不需要手动查询 DOM，再修改字数元素。React 会观察 state，state 一变就自动刷新依赖它的界面。上一节点击导航切换页面，其实也已经使用了同一种机制：App 保存“当前在哪一页”的 state。

## 用 URL 管理路由

上一节的 App 可以通过 state 切换页面，这种页面切换叫路由。但只把路由保存在 state 中有明显问题：

- 页面切换后地址栏不变。
- 刷新页面时，浏览器内存被清空，页面会回到首页。
- 浏览器前进、后退无法正确记录页面切换。

原因是 state 保存在浏览器内存里，刷新就会丢失。更适合作为路由的数据位置是 URL：

- 点击“文字实验室”后，地址在首页的 / 后添加了 text-lab。
- 复制 /text-lab 给别人，打开后直接进入文字实验室。
- 浏览器后退键可以回到首页。

课程 Demo 中是用数据驱动界面的方式实现的监听 URL 和导航变化。真实项目一般使用现成的 react-router 库，不需要手搓完整路由系统。

## 升级现有项目

课程提供的 zero-to-tech-4-4 Demo 已经把页面数据和路由升级好了。可以先运行 Demo 对照现象：

```bash
git clone https://github.com/joylibo/zero-to-tech-demos
cd zero-to-tech-demos/zero-to-tech-4-4
npm install
npm run dev
```

在自己的 zero-to-tech 项目中，按下面的清单从 `zero-to-tech-4-4` 复制或覆盖文件：

### 新增文件

```text
src/data/site.js
src/router/useRoute.js
```

site.js 是集中管理文案的内容表；useRoute.js 负责监听 URL、处理导航和页面切换。

### 用更新后的版本覆盖

```text
src/App.jsx
src/components/HomePage.jsx
src/components/TextLabPage.jsx
src/components/InputCard.jsx
src/css/lab.css
```

InputCard 中新增了“已输入 N 字”的显示，lab.css 需要加入对应的 .lab-count 样式。复制完成后运行：

```bash
npm run dev
```

## 把它发布到公网

4.1 的 Vanilla 项目可以把 Git 拉下来的源码目录直接交给 Nginx。现在项目是 Vite + React 工程，浏览器不认识 .jsx，必须先构建；而 dist/ 又被 .gitignore 忽略，不会随源代码推送到 GitHub。

因此服务器需要先安装 Node.js，再安装项目依赖并执行 npm run build，最后让 Nginx 指向构建出来的 dist/。流程比 4.1 多了一步构建，但换来了组件化和工程化能力。

### 第一步：给服务器安装 Node.js

SSH 远程登录 Ubuntu 服务器后，全局安装 Node.js 命令：

```bash

# 导入 node 22版本的软件源，且自动 update
curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -

sudo apt install nodejs npm -y

# 验证版本
node -v
npm -v
```

### 第二步：推送和拉取代码

本地完成修改后先推送到 GitHub；SSH 登录服务器后，在项目目录拉取最新源代码：

```bash
cd ~/zero-to-tech
git pull
```

这里拉下来的是源码，不是最终给浏览器访问的 dist/。

### 第三步：在服务器安装依赖并构建

```bash
cd ~/zero-to-tech
npm install
npm run build
```

构建完成后会生成：

```text
dist/
├── index.html             页面外壳
└── assets/
    ├── index-[hash].js   打包压缩后的 JavaScript
    └── index-[hash].css  打包压缩后的 CSS
```

dist/ 才是浏览器可以直接加载的构建产物。

### 第四步：让 Nginx 指向 dist/

编辑 Nginx 配置：

```bash
sudo vim /etc/nginx/sites-enabled/default
```

只需要把 root 指向构建产物目录修改为：`root /home/ubuntu/zero-to-tech/dist`;


检查配置并重新加载 Nginx：

```bash
sudo nginx -t
sudo systemctl reload nginx
```

浏览器访问 http://你的服务器IP/，应看到和本地一致的 React 页面。以后每次更新的完整链路是：本地 git push，服务器 git pull，npm install，npm run build，必要时 reload Nginx。